# Filtro 1

## Entreno de modelo

### Importación de librerías

In [ ]:
import os
import numpy as np
import joblib
from deepface import DeepFace
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from tqdm import tqdm
import kagglehub

### Importación de dataset y definición de categorías

In [2]:
DATASET_ID  = "saramhai/people-with-and-without-glasses-dataset"
DOWNLOAD_PATH  = kagglehub.dataset_download(DATASET_ID)

CLASS_LABELS  = ["glasses", "no_glasses"] 
FACE_MODEL = "ArcFace"
MODEL_FILENAME = "glasses_classifier.pkl"
LABELS_FILE = "glasses_labels.pkl"

embeddings = []
labels = []


### Entrenamiento del dataset

In [ ]:
base_path = os.path.join(DOWNLOAD_PATH , "images")

for label_id, label_name in enumerate(CLASS_LABELS):
    label_path = os.path.join(base_path, label_name)
    if not os.path.exists(label_path):
        print(f"¡Aviso! No se encontró la carpeta: {label_path}")
        continue

    img_files = [f for f in os.listdir(label_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Mezclar aleatoriamente
    np.random.shuffle(img_files)

    print(f"\nProcesando categoría: {label_name} (Usando {len(img_files)} imágenes)")

    # Extraer embeddings
    for filename in tqdm(selected_files, desc=label_name):
        file_path = os.path.join(label_path, filename)
        try:
            rep_data = DeepFace.represent(
                img_path=file_path,
                model_name=FACE_MODEL,
                enforce_detection=False,
                detector_backend='skip'
            )

            face_vector = rep_data[0]["embedding"]
            embeddings.append(face_vector)
            labels.append(label_id)

        except Exception as err:
            if 'face could not be detected' not in str(err):
                print(f"Error procesando {file_path}: {err}")

if not embeddings:
    print("Error: No se extrajo ningún embedding. Verifica el dataset.")
    exit()

embeddings = np.array(embeddings)
labels = np.array(labels)

print(f"\nTotal de embeddings extraídos: {embeddings.shape[0]}")
print(f"Dimensiones de cada embedding: {embeddings.shape[1]}")
print(f"Distribución de clases: {np.bincount(labels)}")

# Evaluación con train/test split
print("\nEvaluando modelo con split 70/30...")
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.3, random_state=42, stratify=labels
)

temp_model = SVC(kernel='linear', probability=True, random_state=42)
temp_model.fit(X_train, y_train)

y_pred = temp_model.predict(X_test)
print("\n" + classification_report(y_test, y_pred, target_names=CLASS_LABELS))

# Entrenamiento final con todos los datos
print("\nEntrenando modelo final con todos los datos...")
best_model = SVC(kernel='linear', probability=True, random_state=42)
best_model.fit(embeddings, labels)

# Guardar modelo y categorías
joblib.dump(best_model, MODEL_FILENAME)
joblib.dump(CLASS_LABELS, LABELS_FILE)

print(f"\nModelo: '{MODEL_FILENAME}'")
print(f"Categorías: '{LABELS_FILE}'")

### Uso

In [ ]:
import cv2
import numpy as np
import joblib
from deepface import DeepFace

# --- Configuración ---
MODEL_FILENAME = "glasses_classifier.pkl"
CATEGORIES_FILENAME = "glasses_labels.pkl"
FACE_MODEL = "ArcFace"  # ¡Debe ser el mismo que en el entrenamiento!

# --- Cargar modelos ---
try:
    model = joblib.load(MODEL_FILENAME)
    categories = joblib.load(CATEGORIES_FILENAME)
except FileNotFoundError:
    print(f"Error: No se encontraron los archivos '{MODEL_FILENAME}' o '{CATEGORIES_FILENAME}'.")
    print("Asegúrate de ejecutar el script de entrenamiento primero.")
    exit()

# Detector de rostros de OpenCV (para encontrar la cara RÁPIDO)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: No se pudo abrir la cámara.")
    exit()

print("Presiona 'q' para salir...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Espejar la imagen para que parezca un espejo
    frame = cv2.flip(frame, 1)
    output_frame = frame.copy()
    
    # Usar el detector de OpenCV (rápido) para encontrar la cara
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

    for (x, labels, w, h) in faces:
        # Recortar el rostro (en BGR, DeepFace prefiere BGR)
        face_img = frame[labels:labels+h, x:x+w]

        # Evitar errores si el recorte está vacío
        if face_img.size == 0:
            continue

        try:
            # 1. Extraer embedding con DeepFace (el mismo modelo de entrenamiento)
            # Usamos 'sQQkip' porque ya le estamos pasando una cara recortada
            embedding_objs = DeepFace.represent(
                img_path=face_img,
                model_name=FACE_MODEL,
                enforce_detection=False, # No es necesario que DeepFace detecte de nuevo
                detector_backend='skip'  # ¡Importante!
            )
            
            # 2. Preparar el embedding para el modelo SVM
            embedding_vector = embedding_objs[0]["embedding"]
            embedding_data = np.array(embedding_vector).reshape(1, -1)

            # 3. Predecir con el modelo SVM
            prediction_idx = model.predict(embedding_data)[0]
            prediction_proba = model.predict_proba(embedding_data)[0]
            
            label = categories[prediction_idx]
            confidence = prediction_proba[prediction_idx] * 100

            # 4. Dibujar resultados
            color = (0, 255, 0) if label == "no_glasses" else (0, 0, 255)
            text = f"{label} ({confidence:.1f}%)"
            
            cv2.rectangle(output_frame, (x, labels), (x+w, labels+h), color, 2)
            cv2.putText(output_frame, text, (x, labels-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        except Exception as e:
            # DeepFace puede fallar si la cara es muy pequeña o está borrosa
            # print(f"Error de DeepFace: Qqqqq{e}") # Descomentar para depurar
            pass

    # Mostrar resultado
    cv2.imshow("Deteccion de Gafas (DeepFace + SVM)", output_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --- Limpieza ---
cap.release()
cv2.destroyAllWindows()